# A1 — Data pipeline setup

Builds every intermediate dataset the analysis depends on, in dependency
order. Each step skips itself if its output already exists, so the notebook
is safe to re-run.

**Run this before B1, B2 or C1.** Those notebooks read the files produced
here and will fail or return stale results if the pipeline has not been run.

**If you change a script in `src/pipeline/`, you must re-run its step here.**
The analysis notebooks read Parquet files, not code — editing a script has no
effect until the corresponding rebuild is executed. Delete the relevant
output directory first, since each step skips when its output is present.

Expect this to take several hours from scratch, dominated by the CBOE
extraction and the moneyness join over ~145 million option-level rows.

## Setup — resolve project paths

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    """Walk upward from the current directory until a 'src' folder is found."""
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
from paths import PROJECT_ROOT, ZIP_DIR, DATA_DIR, EXTRACTED_DIR, PARQUET_DIR, IBES_DIR, CRSP_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL


## Step 1 — CBOE options data

Extracts the per-day zip archive to CSV, then converts to typed, compressed
yearly Parquet files.

In [2]:
from pipeline.extract_zips import run_extraction
from pipeline.ingest_cboe import run_ingestion

if not any(PARQUET_DIR.glob("*.parquet")):
    run_extraction()
    run_ingestion(raw_dir=str(EXTRACTED_DIR), out_dir=str(PARQUET_DIR))
else:
    print("Parquet files already exist -- skipping. "
          "Delete data/cboe_parquet/ first if you want to rebuild.")

Parquet files already exist -- skipping. Delete data/cboe_parquet/ first if you want to rebuild.


## Step 2 — Verify CBOE output

Row counts, disk usage, and a spot check of one day's Parquet output against
a fresh read of its source zip.

In [3]:
from pipeline.verify_setup import main
main()

=== Yearly Parquet row counts ===
  cboe_openclose_2011.parquet: 7,587,825 rows
  cboe_openclose_2012.parquet: 7,429,806 rows
  cboe_openclose_2013.parquet: 7,849,786 rows
  cboe_openclose_2014.parquet: 10,045,109 rows
  cboe_openclose_2015.parquet: 8,900,411 rows
  cboe_openclose_2016.parquet: 9,324,066 rows
  cboe_openclose_2017.parquet: 10,834,820 rows
  cboe_openclose_2018.parquet: 13,921,941 rows
  cboe_openclose_2019.parquet: 14,774,622 rows
  cboe_openclose_2020.parquet: 20,826,043 rows
  cboe_openclose_2021.parquet: 24,336,763 rows
  cboe_openclose_2022.parquet: 9,047,229 rows
  Total: 144,878,421 rows across 12 file(s)

=== Disk usage ===
  Raw zip archives: 4.19 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\CBOE_Data_2011_2022
  Parquet output: 3.95 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\cboe_parquet
  IBES data: 0.68 GB -- C:\Users\axels\CBOE_DATA_ANALYSIS_ASPARN_FINAL\data\ibes_quarterly_report
  Free space: 40.7 GB

=== Spot check against a fresh 

## Step 3 — IBES analyst forecasts

Detects the delimiter, parses dates explicitly, and fails loudly if the date
format does not match — an earlier version of this pipeline silently wrote
null dates for an entire file.

In [4]:
from pipeline.ingest_ibes import run_ibes_ingestion

if not (IBES_DIR / "ibes_clean.parquet").exists():
    run_ibes_ingestion()
else:
    print("ibes_clean.parquet already exists -- skipping.")

ibes_clean.parquet already exists -- skipping.


## Step 4 — Firm-event dispersion panel

One row per (ticker, fiscal quarter) earnings event, carrying analyst
forecast dispersion and prior earnings volatility. Both are scaled and
winsorised; see the module docstring for why.

In [5]:
from pipeline.build_dispersion_events import build_dispersion_events

if not (IBES_DIR / "dispersion_events.parquet").exists():
    build_dispersion_events()
else:
    print("dispersion_events.parquet already exists -- skipping.")

dispersion_events.parquet already exists -- skipping.


## Step 5 — Daily participant activity

Collapses option-contract-level data to one row per (ticker, day), split by
participant type. Retail and professional customers get contract-size tiers;
firm, broker-dealer and market-maker flow is reported without size
breakdowns, so only totals are available for those.

In [6]:
from pipeline.build_daily_retail_activity import build_daily_retail_activity

daily_dir = DATA_DIR / "cboe_daily_retail"
if not list(daily_dir.glob("*.parquet")):
    build_daily_retail_activity()
else:
    print("daily_retail_*.parquet already exists -- skipping. "
          "Delete the folder first if you've changed build_daily_retail_activity.py.")

daily_retail_*.parquet already exists -- skipping. Delete the folder first if you've changed build_daily_retail_activity.py.


## Step 6 — CRSP daily prices

Streams the ~3.7GB WRDS export to Parquet. Provides spot prices for
moneyness classification and market capitalisation for the firm-size
control.

In [7]:
from pipeline.ingest_crsp import run_crsp_ingestion

if not (CRSP_DIR / "crsp_daily.parquet").exists():
    run_crsp_ingestion()
else:
    print("crsp_daily.parquet already exists -- skipping. "
          "Delete it first if you need to re-ingest.")

crsp_daily.parquet already exists -- skipping. Delete it first if you need to re-ingest.


## Step 7 — Moneyness classification

Joins CRSP spot prices to option-level CBOE data and classifies every traded
contract as out-of-the-money, in-the-money or at-the-money. This is the
slowest step, since moneyness must be computed per contract rather than from
the aggregated daily table.

In [8]:
from pipeline.build_moneyness import build_moneyness

mny_dir = DATA_DIR / "cboe_daily_moneyness"
if not list(mny_dir.glob("*.parquet")):
    build_moneyness()
else:
    print("daily_moneyness_*.parquet already exists -- skipping. "
          "Delete the folder first if you've changed build_moneyness.py.")

daily_moneyness_*.parquet already exists -- skipping. Delete the folder first if you've changed build_moneyness.py.


## Step 8 — Verify moneyness coverage

The event-sample percentage is what certifies DV2 is usable. Unpriced volume
outside the event sample is index and volatility products (^SPX, ^VIX and
similar), which have no earnings announcements and were never in the sample.

In [9]:
from analysis.check_moneyness_coverage import check_coverage
top_unknown = check_coverage()

=== ALL CBOE tickers ===
  Classified retail volume: 7,320,638,834
  Unclassified (no spot):   3,518,238,670  (32.5%)

=== Tickers carrying the most unclassified volume ===
shape: (15, 2)
┌───────────────────┬─────────────┐
│ underlying_symbol ┆ unknown_vol │
│ ---               ┆ ---         │
│ str               ┆ i64         │
╞═══════════════════╪═════════════╡
│ ^SPX              ┆ 2014600713  │
│ ^VIX              ┆ 1137984205  │
│ VXX               ┆ 145523648   │
│ ^RUT              ┆ 101363947   │
│ ^XSP              ┆ 39512461    │
│ …                 ┆ …           │
│ VXXB              ┆ 2675188     │
│ BRK/B             ┆ 2238135     │
│ BRK.B             ┆ 2156873     │
│ WFT               ┆ 1892344     │
│ ITUB              ┆ 1615957     │
└───────────────────┴─────────────┘

Event-sample tickers: 5,509

=== EVENT-SAMPLE tickers only (the number that matters) ===
  Classified retail volume: 4,633,377,287
  Unclassified (no spot):   9,135,808  (0.2%)


## Step 9 — Verify headline results

Recomputes every number reported in the thesis from the current state of the
data and compares against the reported values. All checks should read MATCH.

A mismatch means a result was computed against a superseded intermediate
file and needs revisiting before it is relied on.

In [10]:
from analysis.verify_results import verify
verify()

VERIFYING HEADLINE RESULTS

1. Sample construction (full period)
  firm-events in dispersion panel                     163,010  expected      163,010  MATCH
  matched to CBOE trading data                         97,270  expected       97,270  MATCH

2. Binary difference-in-differences (balanced panel, 2016-01-01 onward)
  otm treat:post                                      +0.0388  expected      +0.0388  MATCH
  otm_put treat:post                                  +0.0226  expected      +0.0226  MATCH
  otm_call treat:post                                 +0.0162  expected      +0.0162  MATCH
  itm treat:post                                      -0.0170  expected      -0.0170  MATCH
  lt_100 treat:post                                   -0.0133  expected      -0.0133  MATCH
  call treat:post                                     -0.0189  expected      -0.0189  MATCH
  open treat:post                                     +0.0184  expected      +0.0184  MATCH

3. Analyst dispersion, cross-sect

np.True_